In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from propofol.haemo_pd_su2023 import SuHaemoPD
from propofol.patient import EleveldPatient as Patient
from propofol.propofol_pkpd import EleveldPK as PropofolPK
from propofol.propofol_pkpd import EleveldPD as PropofolPD
from propofol.remifentanil_pkpd import EleveldPK as RemifentanilPK


# ============================================================
# 1. Simulation settings
# ============================================================
N_SIM = 1000
AGES = [22, 47, 60]
WEIGHT = 70.0
HEIGHT = 180.0
SEX = "male"

T_END = 120.0
DT = 1.0 / 60.0  # 1 second in minutes
TIME = np.arange(0.0, T_END + DT, DT)

INDUCTION_DURATION_MIN = 1.0
MAINTENANCE_END_MIN = 60.0

PROPOFOL_INDUCTION_MGKG = 1.5
PROPOFOL_MAINT_MGKGMIN = 0.1
REMI_MAINT_UGKGMIN = 0.1

LOW_Q = 5
MID_Q = 50
HIGH_Q = 95


# ============================================================
# 2. Dosing objects
# ============================================================
class PiecewiseInfusion:
    def __init__(self, segments):
        self.segments = segments
        crit = set()
        for t0, t1, _ in segments:
            crit.add(float(t0))
            crit.add(float(t1))
        self.tcrit = sorted(crit)

    def dotA0(self, t):
        for t0, t1, rate in self.segments:
            if t0 <= t < t1:
                return rate
        return 0.0


def make_propofol_dosing(weight_kg):
    induction_total_mg = PROPOFOL_INDUCTION_MGKG * weight_kg
    induction_rate_mgmin = induction_total_mg / INDUCTION_DURATION_MIN
    maintenance_rate_mgmin = PROPOFOL_MAINT_MGKGMIN * weight_kg

    segments = [
        (0.0, INDUCTION_DURATION_MIN, induction_rate_mgmin),
        (INDUCTION_DURATION_MIN, MAINTENANCE_END_MIN, maintenance_rate_mgmin),
        (MAINTENANCE_END_MIN, T_END, 0.0),
    ]
    return PiecewiseInfusion(segments)


def make_remifentanil_dosing(weight_kg):
    rate_ugmin = REMI_MAINT_UGKGMIN * weight_kg
    rate_mgmin = rate_ugmin / 1000.0

    segments = [
        (0.0, MAINTENANCE_END_MIN, rate_mgmin),
        (MAINTENANCE_END_MIN, T_END, 0.0),
    ]
    return PiecewiseInfusion(segments)


# ============================================================
# 3. One simulation
# ============================================================
def run_one_sim(age, weight, height, sex):
    patient = Patient(
        age=age,
        height=height,
        weight=weight,
        sex=sex,
        opiates=False,
    )

    pk_propofol = PropofolPK(patient, use_bsv=True)
    pd_propofol = PropofolPD(patient, use_bsv=True)
    pk_remifentanil = RemifentanilPK(patient, use_bsv=True)

    model = SuHaemoPD(
        patient=patient,
        pk_propofol=pk_propofol,
        pd_propofol=pd_propofol,
        pk_remifentanil=pk_remifentanil,
        use_bsv=True,
    )

    dosing_prop = make_propofol_dosing(weight)
    dosing_remi = make_remifentanil_dosing(weight)

    (
        A1, A2, A3, Ce_prop,
        A4, A5, A6,
        sv_ast, hr_ast, tpr,
        tde_sv, tde_hr,
        sv, MAP,
    ) = model.solve_ode(
        t=TIME,
        dosing_prop=dosing_prop,
        dosing_remi=dosing_remi,
    )

    cp_prop = A1 / model.pk_propofol.V1
    cp_remi = (A4 / model.pk_remifentanil.V1) * 1000.0  # ng/mL
    hr = hr_ast + tde_hr

    map0 = float(MAP[0])
    hr0 = float(hr[0])
    sv0 = float(sv[0])

    map_pct = 100.0 * (MAP - map0) / map0
    hr_pct = 100.0 * (hr - hr0) / hr0
    sv_pct = 100.0 * (sv - sv0) / sv0

    return {
        "time": TIME,
        "cp_prop": cp_prop,
        "cp_remi": cp_remi,
        "MAP": MAP,
        "HR": hr,
        "SV": sv,
        "map_pct": map_pct,
        "hr_pct": hr_pct,
        "sv_pct": sv_pct,
    }


# ============================================================
# 4. Batch simulation
# ============================================================
def run_batch(age, n_sim=N_SIM):
    cp_prop_all = []
    cp_remi_all = []
    map_pct_all = []
    hr_pct_all = []
    sv_pct_all = []

    for _ in range(n_sim):
        sim = run_one_sim(age=age, weight=WEIGHT, height=HEIGHT, sex=SEX)
        cp_prop_all.append(sim["cp_prop"])
        cp_remi_all.append(sim["cp_remi"])
        map_pct_all.append(sim["map_pct"])
        hr_pct_all.append(sim["hr_pct"])
        sv_pct_all.append(sim["sv_pct"])

    return {
        "time": TIME,
        "cp_prop": np.asarray(cp_prop_all),
        "cp_remi": np.asarray(cp_remi_all),
        "map_pct": np.asarray(map_pct_all),
        "hr_pct": np.asarray(hr_pct_all),
        "sv_pct": np.asarray(sv_pct_all),
    }


def summarize_band(x):
    return {
        "p5": np.percentile(x, LOW_Q, axis=0),
        "p50": np.percentile(x, MID_Q, axis=0),
        "p95": np.percentile(x, HIGH_Q, axis=0),
    }


# ============================================================
# 5. Run all ages
# ============================================================
results = {}
for age in AGES:
    print(f"Running {N_SIM} simulations for age {age}...")
    batch = run_batch(age=age, n_sim=N_SIM)
    results[age] = {
        "time": batch["time"],
        "cp_prop": summarize_band(batch["cp_prop"]),
        "cp_remi": summarize_band(batch["cp_remi"]),
        "map": summarize_band(batch["map_pct"]),
        "hr": summarize_band(batch["hr_pct"]),
        "sv": summarize_band(batch["sv_pct"]),
    }


# ============================================================
# 6. Plot helper
# ============================================================
def plot_outcome(results, outcome_key, ylabel):
    fig, axes = plt.subplots(1, len(AGES), figsize=(5 * len(AGES), 4.5), sharex=True, sharey=True)

    if len(AGES) == 1:
        axes = [axes]

    for ax, age in zip(axes, AGES):
        t = results[age]["time"]
        band = results[age][outcome_key]

        ax.fill_between(t, band["p5"], band["p95"], alpha=0.25)
        ax.plot(t, band["p50"], lw=2)

        ax.set_title(f"Age {age} yr")
        ax.set_xlabel("Time (min)")
        ax.set_ylabel(ylabel)

    fig.tight_layout()
    plt.show()


# ============================================================
# 7. Plot PK
# ============================================================
plot_outcome(
    results=results,
    outcome_key="cp_prop",
    ylabel="Propofol plasma concentration (µg/mL)",
)

plot_outcome(
    results=results,
    outcome_key="cp_remi",
    ylabel="Remifentanil plasma concentration (ng/mL)",
)


# ============================================================
# 8. Plot haemodynamics
# ============================================================
plot_outcome(
    results=results,
    outcome_key="map",
    ylabel="MAP change from baseline (%)",
)

plot_outcome(
    results=results,
    outcome_key="hr",
    ylabel="HR change from baseline (%)",
)

plot_outcome(
    results=results,
    outcome_key="sv",
    ylabel="SV change from baseline (%)",
)